# Steerling-8B: Concept Steering

**Concept steering** amplifies a target concept during generation, with no weight updates. Steerling does this at inference time by injecting the concept direction into its hidden state.

The public steering recipe is `injection`. It adds the concept direction to the residual stream (the running hidden state) at the deeper layers, pushing generation toward the concept. Injection strength reads in logit units: it is scaled so `mai_lm_target` is roughly the boost applied to the concept's top-aligned token.

Steering uses a **hard_cutoff** schedule. Full strength applies for the first `cutoff_tokens` generated tokens, then drops to zero. This plants the concept early and lets the rest generate unsteered, which keeps the text fluent.

This notebook generates twice on the same neutral prompt, a reference and a steered run, then checks that the target concept's activation rises.

**Requirements:** an interpretable Steerling checkpoint and a GPU with >= 18 GB VRAM.

## 1. Load the model

Steering needs the **interpretable** model, which exposes the concept heads the injection is built from. We load via HuggingFace `AutoModel` and wrap it in `SteerlingGenerator`.

In [1]:
import sys
sys.path.insert(0, "/fss/nguyeng/june-release/steerling")

import torch
from steerling import SteerlingGenerator, GenerationConfig, SteeringConfig
from steerling.configs.causal_diffusion import CausalDiffusionConfig
from steerling.configs.concept import ConceptConfig
from steerling.data.tokenizer import SteerlingTokenizer
from steerling.models.interpretable.interpretable_causal_diffusion import InterpretableCausalDiffusionLM
from steerling.inference.checkpoint_utils import load_config, load_state_dict

model_id = "guidelabs/steerling-8b"

raw = load_config(model_id)
model_cfg = CausalDiffusionConfig.model_validate(
    {k: v for k, v in raw.items() if k in CausalDiffusionConfig.model_fields}
)
concept_fields = set(ConceptConfig.model_fields)
concept_cfg = ConceptConfig.model_validate({k: v for k, v in raw.items() if k in concept_fields})
vocab_size = raw.get("vocab_size")

model = InterpretableCausalDiffusionLM(model_cfg, concept_cfg, vocab_size)
state = load_state_dict(model_id)
model.load_state_dict(state, strict=False)
if hasattr(model, "transformer"):
    model.transformer._restore_weight_tying()
model = model.to(dtype=torch.bfloat16)

tokenizer = SteerlingTokenizer(instruct=raw.get("instruct", False))
generator = SteerlingGenerator(
    model=model,
    tokenizer=tokenizer,
    model_config=model_cfg,          # the CausalDiffusionConfig you built above
    is_interpretable=True,
    device="cuda",
)

# flex attention needs to be wired, which from_pretrained/from_model normally do:
import steerling.models.layers.causal_diffusion_layers as layers
from torch.nn.attention.flex_attention import flex_attention
layers.compiled_flex_attention = flex_attention

print(type(generator.model).__name__)              # InterpretableCausalDiffusionLM
print(hasattr(generator.model, "_forward_with_injection"))  # True
assert generator.is_interpretable

InterpretableCausalDiffusionLM
True


## 2. Choose a concept and a neutral prompt

Set the target concept and a **neutral** prompt, one that does not mention the concept. Steering should weave the concept into the continuation that would otherwise be generic.

Set `CONCEPT_ID`, `CONCEPT_LABEL`, and `CONCEPT_DESCRIPTION` to a concept from your concept table.

In [6]:
# --- target concept (replace with one from your concept table) ---
CONCEPT_ID = 4524
CONCEPT_LABEL = "Book & Publishing "
# CONCEPT_DESCRIPTION = "Tokens promoted center tightly on cats and feline behavior — explicit cat vocabulary (cat, Cats, Kitty, Kitt, kitten, feline, Bengal, Persian, lynx), anatomy (claws, whiskers, tail, tongue, cheeks, chin, belly), characteristic actions and signals (purr, rubbing, twitch, flick, blinking, curl, stalking, swipe, lounging, sleeps, mating), and a strong sub-theme of feline body-language communication (communicates, signifies, gesture, expressive, signaling, demeanor, posture, gaze, subtle, territorial)."

# --- neutral prompt (no mention of the concept) ---
PROMPT = "Today lets talk about"

# --- generation budget ---
MAX_NEW_TOKENS = 128
STEPS = 128

# --- steering strength ---
# mai_lm_target reads in logit units (positive amplifies). 7 is the preset default.
ALPHA = 8.0
# hard_cutoff: tokens generated at full strength before injection stops.
CUTOFF_TOKENS = 32
# inject_layer=None resolves to n_layers // 2 inside the generator.
INJECT_LAYER = 8

print(PROMPT)

Today lets talk about


## 3. Reference generation

First generate with no intervention. This is the baseline the steered run is compared against. Reference sampling uses `temperature=1.2`, `top_p=0.8`, `repetition_penalty=1.1`.

In [7]:
ref_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=1.2,
    top_p=0.8,
    repetition_penalty=1.1,
    seed=0,
)

reference = generator.generate_full(PROMPT, ref_config)
print(reference.text)

 what is a Parallax and how to use it in your website. You might have seen a lot of websites using parallax scrolling effect. It's really amazing, isn't it? A parallax is an illusion that background elements move slower than foreground elements.

Parallax effect can be used for different purposes such as navigation bar, hero section or landing page. I will show you how to create a simple parallax scroll with HTML and CSS. We are going to use Bootstrap 4.3.1 for the demonstration. So first of all let's import the necessary libraries and set some stylesheets:

The next thing we need


## 4. Steer toward the concept (`injection`)

`SteeringConfig.injection` builds the steering config: positive injection of the concept direction on the hard_cutoff schedule. We pass it to `generate_steered`. Steered sampling uses `temperature=0.9`, `top_p=0.7`, `repetition_penalty=1.5`.

In [8]:
steered_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=0.9,
    top_p=0.7,
    repetition_penalty=1.5,
    seed=0,
)

steer = SteeringConfig.injection(
    concept_ids=CONCEPT_ID,
    mai_lm_target=ALPHA,
    inject_layer=INJECT_LAYER,
    cutoff_tokens=CUTOFF_TOKENS,
)

steered = generator.generate_steered(PROMPT, steered_config, steer)
print(steered.text)

 ISBN.

ISBN (International Standard Book Number) is a series of numbers used to identify books and other publications. The CRC Press imprint of Cengage Learning, Incorporated / John Wiley & Sons has an internal numbering system that uses the 13-digit International Standards Books Numbers(10).

The international standard book number was introduced in as part of ISO/IEC It consists of three parts: An identifier for the publisher; A code indicating what type of publication it represents - e.g., paper edition or electronic version;

A check digit which identifies whether this title belongs within one of several groups defined by publishers' rights societies such ASINs are assigned


In [9]:
import torch
cid = torch.tensor([4524], device=generator.device)
emb = generator.model.known_head._get_embedding(cid).float()      # (1, D)
d = emb.sum(0); direction = d / (d.norm() + 1e-12)
lm_w = generator.model.transformer.lm_head.weight.float()         # (V, D)
align = lm_w @ direction
peak = align.max().item()
print("peak:", peak, "| base_alpha = ALPHA/peak =", ALPHA / peak)
topv, topi = align.topk(15)
print("top-aligned tokens:", [generator.tokenizer.decode([int(t)]) for t in topi])

peak: 0.90545254945755 | base_alpha = ALPHA/peak = 8.835360842258096
top-aligned tokens: ['ISBN', ' ISBN', 'Publisher', ' Rout', '978', ' Penguin', ' Press', ' Wiley', ' isbn', 'ledge', 'mill', 'enguin', 'isbn', ' Sons', 'court']
